In [1]:
import numpy as np
import pandas as pd

In [2]:
import torch

In [3]:
df=pd.read_csv("final_train_dataset.csv")

In [4]:
df.head()

,ats_score,resume_text,job_description
0,80.6,SummaryI am seeking a position wherein I may u...,- Share resume to shan imrsoft.com Job Title J...
1,24.3,ProfileHighly motivated Sales Associate with e...,Purpose StatementThe Software Engineering Mana...
2,53.9,SummaryHaving achieved a milestone of 7 years ...,"Title Business AnalystLocation Santa Clara, CA..."
3,52.5,SummaryWireless communications engineer with e...,Job Description As a Principal Software Engine...
4,59.2,SummaryData Entry experienced and adept at inp...,"Title Data EngineerLocation fully remote, howe..."


In [5]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

In [6]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 4079 entries, 3497 to 860
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ats_score        4079 non-null   float64
 1   resume_text      4079 non-null   str    
 2   job_description  4079 non-null   str    
dtypes: float64(1), str(2)
memory usage: 33.0 MB


In [7]:
val_df.info()

<class 'pandas.DataFrame'>
Index: 1020 entries, 996 to 3944
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ats_score        1020 non-null   float64
 1   resume_text      1020 non-null   str    
 2   job_description  1020 non-null   str    
dtypes: float64(1), str(2)
memory usage: 8.4 MB


In [8]:
train_df["label"] = train_df["ats_score"] / 100
val_df["label"] = val_df["ats_score"] / 100

In [9]:
from sentence_transformers import InputExample
InputExample(
    texts=[ "resume_text","job_description"],
    label=0.82   ## eXPECTED lABEL
)

In [10]:
from sentence_transformers import InputExample

train_examples = [
    InputExample(
        texts=[row["resume_text"], row["job_description"]],
     label=float(row["label"])
    )
    for _, row in train_df.iterrows()
]

val_examples = [
    InputExample(
        texts=[row["resume_text"], row["job_description"]],
         label=float(row["label"])
    )
    for _, row in val_df.iterrows()
]


In [11]:
from torch.utils.data import DataLoader

In [12]:
BATCH_SIZE = 16

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE
)

val_dataloader = DataLoader(
    val_examples,
    shuffle=False,
    batch_size=BATCH_SIZE
)

## Data model loading

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
from sentence_transformers.losses import CosineSimilarityLoss

C:\Users\Shreyas\AppData\Local\Temp\ipykernel_2520\363032932.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import CosineSimilarityLoss


In [16]:
train_loss = CosineSimilarityLoss(model)

In [17]:
import sentence_transformers
print(sentence_transformers.__version__)

5.6.0


In [18]:
import datasets
num_epochs = 3

warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path="sbert_resume_matcher",
    show_progress_bar=True
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.046040


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
from sentence_transformers import SentenceTransformer
import torch

model = SentenceTransformer("sbert_resume_matcher")
model.eval()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [30]:
from sentence_transformers.util import cos_sim

predictions = []
actual = []

for _, row in val_df.iterrows():
    resume_emb = model.encode(row["resume_text"], convert_to_tensor=True)
    jd_emb = model.encode(row["job_description"], convert_to_tensor=True)

    similarity = cos_sim(resume_emb, jd_emb).item()

    predictions.append(similarity * 100)  
    actual.append(row["ats_score"])

In [31]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(actual, predictions)
rmse = np.sqrt(mean_squared_error(actual, predictions))

print("MAE :", mae)
print("RMSE:", rmse)

MAE : 16.228549105024804
RMSE: 20.068348524741573


In [32]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

base_model = SentenceTransformer("all-MiniLM-L6-v2")

base_predictions = []

for _, row in val_df.iterrows():
    resume_emb = base_model.encode(row["resume_text"], convert_to_tensor=True)
    jd_emb = base_model.encode(row["job_description"], convert_to_tensor=True)

    similarity = cos_sim(resume_emb, jd_emb).item()
    base_predictions.append(similarity * 100)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [33]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr
import numpy as np

mae = mean_absolute_error(actual, base_predictions)
rmse = np.sqrt(mean_squared_error(actual, base_predictions))
corr, _ = pearsonr(actual, base_predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("Pearson:", corr)

MAE: 20.862226670212312
RMSE: 25.496952556152085
Pearson: 0.23638506639828727


### validating with given validation dataset

In [24]:
new_val_df=pd.read_csv("final_validation_dataset.csv")

In [25]:
new_val_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1275 entries, 0 to 1274
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ats_score        1275 non-null   float64
 1   resume_text      1275 non-null   str    
 2   job_description  1275 non-null   str    
dtypes: float64(1), str(2)
memory usage: 10.2 MB


In [28]:
from sentence_transformers.util import cos_sim

final_valid_predictions = []
final_valid_actual = []

for _, row in new_val_df.iterrows():
    resume_emb = model.encode(row["resume_text"], convert_to_tensor=True)
    jd_emb = model.encode(row["job_description"], convert_to_tensor=True)

    similarity = cos_sim(resume_emb, jd_emb).item()

    final_valid_predictions.append(similarity * 100)  
    final_valid_actual.append(row["ats_score"])

In [29]:
mae = mean_absolute_error(final_valid_actual , final_valid_predictions)
rmse = np.sqrt(mean_squared_error(final_valid_actual , final_valid_predictions))
corr, _ = pearsonr(final_valid_actual ,final_valid_predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("Pearson:", corr)

MAE: 16.501888030026002
RMSE: 20.52603050547194
Pearson: 0.585178665509845


### inference

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [4]:
model = SentenceTransformer("sbert_resume_matcher")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
def score_resume_against_jd(resume_text, job_description):
    """
    Compute semantic similarity between a resume and job description.
    """

    resume_embedding = model.encode(
        resume_text,
        convert_to_numpy=True
    )

    jd_embedding = model.encode(
        job_description,
        convert_to_numpy=True
    )

    similarity = cosine_similarity(
        [resume_embedding],
        [jd_embedding]
    )[0][0]

    return float(similarity)

In [6]:
resume = """
Python Developer with FastAPI, SQL, NLP and Machine Learning experience.
"""

jd = """
Looking for a Python Developer with NLP, SQL and FastAPI skills.
"""

score = score_resume_against_jd(resume, jd)

print(f"Similarity Score: {score:.4f}")

Similarity Score: 0.9070
